In [0]:
# ============================================================
# Pipeline : covid19_medallion_pipeline
# Notebook : 04_silver/02_silver_encounters
# Purpose  : Encounters + Quarantine
#
# The temporal validation rule evolved from a simple defensive check (STOP >= START)
# to precise dynamic thresholds per encounter_class — after discovering actual
# outliers (via a query on fact_encounters post-build) with durations up to 105
# years (STOP far exceeding START, yet technically passing the old STOP >= START rule).
#
# The thresholds below are based on actual p99 values extracted from the real data
# for each encounter_class individually (not a uniform theoretical assumption):
#   wellness   -> actual p99 = 47 min       -> safe limit 2880 (2 days)
#   urgentcare -> actual p99 = 15 min       -> safe limit 1440 (1 day)
#   emergency  -> actual p99 = 1440 min     -> safe limit 4320 (3 days, x3 margin)
#   ambulatory -> actual p99 = 20160 min    -> safe limit 20160 (exact match)
#   outpatient -> actual p99 = 84 min only! -> safe limit 600 (10 hours, not 14 days)
#   inpatient  -> actual p99 = 30356 min    -> safe limit 129600 (90 days, wider margin)
#
# Important: This rule is specific to fact_encounters only — it is not generalized
# to medications/conditions/careplans, because a long treatment duration or a chronic
# condition lasting years is perfectly legitimate there, unlike an Encounter which
# is inherently a time-bounded visit by type.
# ============================================================

import dlt
from pyspark.sql import functions as F

TEMPORAL_VALID_RULE_DATETIME = """
stop_datetime IS NULL
OR (
    stop_datetime >= start_datetime
    AND (unix_timestamp(stop_datetime) - unix_timestamp(start_datetime)) / 60 <=
        CASE lower(encounter_class)
            WHEN 'wellness'    THEN 2880
            WHEN 'urgentcare'  THEN 1440
            WHEN 'emergency'   THEN 4320
            WHEN 'ambulatory'  THEN 20160
            WHEN 'outpatient'  THEN 600
            WHEN 'inpatient'   THEN 129600
            ELSE 129600
        END
)
"""


@dlt.table(name="covid19_lakehouse.silver.encounters", comment="Encounters with correct types")
@dlt.expect_or_drop("valid_patient_id", "patient_id IS NOT NULL")
@dlt.expect_or_drop("valid_start_datetime", "start_datetime IS NOT NULL")
@dlt.expect_or_drop("valid_stop_after_start", TEMPORAL_VALID_RULE_DATETIME)
def silver_encounters():
    df = dlt.read_stream("covid19_lakehouse.bronze.encounters")
    return (
        df.select(
            F.col("Id").alias("encounter_id"),
            F.col("PATIENT").alias("patient_id"),
            F.col("ORGANIZATION").alias("organization_id"),
            F.col("PROVIDER").alias("provider_id"),
            F.col("PAYER").alias("payer_id"),
            F.to_timestamp("START").alias("start_datetime"),
            F.to_timestamp("STOP").alias("stop_datetime"),
            F.lower(F.col("ENCOUNTERCLASS")).alias("encounter_class"),
            F.col("CODE").alias("encounter_code"),
            F.col("DESCRIPTION").alias("encounter_description"),
            F.col("BASE_ENCOUNTER_COST").cast("decimal(10,2)").alias("base_encounter_cost"),
            F.col("TOTAL_CLAIM_COST").cast("decimal(10,2)").alias("total_claim_cost"),
            F.col("PAYER_COVERAGE").cast("decimal(10,2)").alias("payer_coverage"),
            F.col("REASONCODE").alias("reason_code"),
            F.col("REASONDESCRIPTION").alias("reason_description"),
            F.col("_ingested_at"),
        )
    )


@dlt.table(name="covid19_lakehouse.quarantine.encounters", comment="Encounters rejected by Silver DQ rules")
def quarantine_encounters():
    df = dlt.read_stream("covid19_lakehouse.bronze.encounters")

    start_ts = F.to_timestamp(F.col("START"))
    stop_ts = F.to_timestamp(F.col("STOP"))

    duration_threshold = F.expr("""
        CASE lower(ENCOUNTERCLASS)
            WHEN 'wellness'    THEN 2880
            WHEN 'urgentcare'  THEN 1440
            WHEN 'emergency'   THEN 4320
            WHEN 'ambulatory'  THEN 20160
            WHEN 'outpatient'  THEN 600
            WHEN 'inpatient'   THEN 129600
            ELSE 129600
        END
    """)
    duration_minutes = (F.unix_timestamp(stop_ts) - F.unix_timestamp(start_ts)) / 60

    is_invalid_patient = F.col("PATIENT").isNull()
    is_invalid_start = F.col("START").isNull() | start_ts.isNull()
    is_invalid_stop_order = stop_ts.isNotNull() & (stop_ts < start_ts)
    is_implausible_duration = (
        stop_ts.isNotNull() & (stop_ts >= start_ts) & (duration_minutes > duration_threshold)
    )

    return (
        df.filter(is_invalid_patient | is_invalid_start | is_invalid_stop_order | is_implausible_duration)
        .select(
            F.lit("encounters").alias("source_table"),
            F.when(is_invalid_patient, "null_patient_id")
             .when(is_invalid_start, "null_or_invalid_start_datetime")
             .when(is_invalid_stop_order, "stop_before_start")
             .when(is_implausible_duration, "duration_exceeds_plausible_threshold_for_encounter_class")
             .otherwise("multiple_dq_violations").alias("rule_violated"),
            F.col("Id").alias("encounter_id"),
            F.col("PATIENT").alias("patient_id"),
            F.col("START").alias("raw_start"),
            F.col("STOP").alias("raw_stop"),
            F.current_timestamp().alias("quarantined_at"),
        )
    )